# **Data Pipeline Notebook**

## **0. Library import**

This section imports the libraries needed for data processing. We import pandas to work with tabular data and the `DataPipeline` class from a custom module to automate the entire BRFSS data processing. We also set the path so that we can use the modules from the `src/` directory.

In [1]:
import os
import sys
# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [2]:
import pandas as pd

from src.pipelines import DataPipeline
from src.config import BRFSS_FILTERING_FILE_PATH

## **1. Download and load dataset**

This section initiates and runs an automated pipeline to download and process the BRFSS data. The pipeline performs a series of steps including downloading raw data from the CDC source, cleaning the data, filtering important features, and standardizing the format. The entire process is logged to a file for monitoring and debugging. `../logs/data_pipeline.log`.

From the log output, we see that the pipeline has successfully downloaded three zip files from CDC `(LLCP2017XPT.zip - 102MB, LLCP2019XPT.zip - 93.4MB, LLCP2021XPT.zip - 77.8MB)`, unzipped, and processed the data. The final result is a combined dataset from 3 years with shape (787,602, 21)—that is, 787,602 records with 21 features.
Once the pipeline is complete, we read the processed data from the final file and display the first five lines for testing.

In [3]:
data_pipeline = DataPipeline(log_file="../logs/data_pipeline.log")
data_pipeline.run_pipeline()

2025-07-27 00:01:40,046 - [src.pipelines] - INFO - DataPipeline initialized successfully
2025-07-27 00:01:40,049 - [src.data] - INFO - BrfssDataFiltering initialized successfully
2025-07-27 00:01:40,049 - [src.pipelines] - INFO - Starting data pipeline processing
2025-07-27 00:01:40,051 - [src.data] - INFO - BrfssDataLoader initialized successfully
2025-07-27 00:01:40,051 - [src.data] - INFO - Starting data loading process...
2025-07-27 00:01:40,052 - [src.data] - INFO - Processing: LLCP2017XPT.zip
2025-07-27 00:01:40,053 - [src.data] - INFO - LLCP2017XPT.zip not found locally. Downloading...
2025-07-27 00:01:40,054 - [src.data] - INFO - Navigating to URL: https://www.cdc.gov/brfss/annual_data/2017/files/LLCP2017XPT.zip
2025-07-27 00:01:40,901 - [src.data] - INFO - Starting download: LLCP2017XPT.zip (104228.63 KB)
LLCP2017XPT.zip: 100%|██████████| 102M/102M [00:04<00:00, 22.2MB/s] 
2025-07-27 00:01:45,706 - [src.data] - INFO - Downloaded: LLCP2017XPT.zip -> /home/qctrung/Projects/indus

In [4]:
# Read combined dataset
df = pd.read_csv(BRFSS_FILTERING_FILE_PATH)
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Year
0,2.0,1.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,1.0,0.0,11.0,6.0,6.0,2017
1,0.0,1.0,0.0,1.0,29.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,10.0,6.0,8.0,2017
2,0.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,14.0,0.0,0.0,10.0,2.0,2.0,2017
3,0.0,1.0,0.0,1.0,27.0,1.0,0.0,1.0,1.0,0.0,...,0.0,3.0,0.0,6.0,0.0,1.0,12.0,4.0,4.0,2017
4,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,1.0,10.0,5.0,8.0,2017


## **2. Overall about the dataset**

### **2.1 Size of the dataset**
From the `df.shape` output, we can see the number of rows (records) and columns (features) in the processed dataset. This helps us evaluate the size of the dataset and see if there is enough data to train complex machine learning models.

In [5]:
# The shape of the dataset
df.shape

(787602, 21)

### **2.2 Previewing the data**
From `df.head()` and `df.tail()`, we see the data structure with features such as Diabetes (target variable), HighBP, HighChol, BMI, Age, GenHlth, etc. All values are numbers (float or int), showing that the data has been well encoded and normalized. The data has a Year column to distinguish the years 2017, 2019, 2021.

In [6]:
# Display the first five rows in the dataset
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Year
0,2.0,1.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,1.0,0.0,11.0,6.0,6.0,2017
1,0.0,1.0,0.0,1.0,29.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,10.0,6.0,8.0,2017
2,0.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,14.0,0.0,0.0,10.0,2.0,2.0,2017
3,0.0,1.0,0.0,1.0,27.0,1.0,0.0,1.0,1.0,0.0,...,0.0,3.0,0.0,6.0,0.0,1.0,12.0,4.0,4.0,2017
4,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,1.0,10.0,5.0,8.0,2017


In [7]:
# Display the last five rows in the dataset
df.tail()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Year
787597,2.0,1.0,1.0,1.0,21.0,0.0,0.0,0.0,1.0,0.0,...,0.0,4.0,0.0,0.0,0.0,1.0,10.0,2.0,3.0,2021
787598,0.0,1.0,0.0,1.0,25.0,1.0,0.0,0.0,1.0,0.0,...,1.0,2.0,20.0,0.0,0.0,0.0,3.0,4.0,5.0,2021
787599,0.0,0.0,1.0,1.0,31.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,7.0,6.0,10.0,2021
787600,0.0,1.0,0.0,1.0,24.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,10.0,4.0,6.0,2021
787601,0.0,0.0,1.0,1.0,32.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,2.0,2.0,0.0,0.0,6.0,6.0,6.0,2021


From `df.columns`, we see that the dataset includes 21 features: Diabetes (target), HighBP, HighChol, CholCheck, BMI, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, Education, Income, Year.

In [8]:
# The columns of dataset
df.columns

Index(['Diabetes', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'HvyAlcoholConsump',
       'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth',
       'DiffWalk', 'Sex', 'Age', 'Education', 'Income', 'Year'],
      dtype='object')

From `df.dtypes`, most of the features are of type float64, only Year is int64. This shows that the data has been processed and encoded into numeric form to be ready for machine learning.

In [9]:
# Data type of each column
df.dtypes

Diabetes                float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
Year                      int64
dtype: object

### **2.3 Dataset information**
From `df.info()`, we see that the dataset has no missing values—all 21 columns have 787,602 non-null entries. This shows that the pipeline has handled missing values well. The dataset takes up about 126.2 MB of memory.

In [10]:
# Display overall about dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787602 entries, 0 to 787601
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes              787602 non-null  float64
 1   HighBP                787602 non-null  float64
 2   HighChol              787602 non-null  float64
 3   CholCheck             787602 non-null  float64
 4   BMI                   787602 non-null  float64
 5   Smoker                787602 non-null  float64
 6   Stroke                787602 non-null  float64
 7   HeartDiseaseorAttack  787602 non-null  float64
 8   PhysActivity          787602 non-null  float64
 9   HvyAlcoholConsump     787602 non-null  float64
 10  AnyHealthcare         787602 non-null  float64
 11  NoDocbcCost           787602 non-null  float64
 12  GenHlth               787602 non-null  float64
 13  MentHlth              787602 non-null  float64
 14  PhysHlth              787602 non-null  float64
 15  

### **2.4 Statistical summary**

From `df.describe().T`, we can draw important insights:

- **Target variable (Diabetes):** Has mean = 0.308, with values from 0 to 2, showing that this is a multi-class problem with 3 classes (0=No diabetes, 1=Pre-diabetes, 2=Diabetes). Most of the data belongs to class 0 (no diabetes).
- **BMI:** Has mean = 28.66, median = 27.0, within the WHO overweight range (25–29.9). Has outliers with max = 99.0.
- **Age:** Encoded into age groups from 1 to 13, with mean = 7.93, median = 8.0, showing that the dataset focuses on the middle-aged group.
- **GenHlth (General Health):** Mean = 2.54, with scale 1–5 (1=excellent, 5=poor), indicating that the majority of participants are in good to great health.
- **MentHlth and PhysHlth:** These are the number of days with health problems in the past month (0–30 days), with means of 3.66 and 4.19 days respectively, indicating that the population is relatively healthy.
- **Binary variables:** HighBP (42.3%), HighChol (39.4%), Smoker (42.5%), PhysActivity (75.4%) indicate the proportion of risk factors in the population.

Overall, the dataset is well-behaved, has no missing values, is large enough for machine learning, and features have a reasonable distribution reflecting the epidemiology of diabetes.

In [11]:
# The distribution of each column
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Diabetes,787602.0,0.307906,0.706731,0.0,0.0,0.0,0.0,2.0
HighBP,787602.0,0.423351,0.494090,0.0,0.0,0.0,1.0,1.0
HighChol,787602.0,0.393967,0.488628,0.0,0.0,0.0,1.0,1.0
CholCheck,787602.0,0.960910,0.193808,0.0,1.0,1.0,1.0,1.0
BMI,787602.0,28.660298,6.412821,12.0,24.0,27.0,32.0,99.0
Smoker,787602.0,0.425391,0.494403,0.0,0.0,0.0,1.0,1.0
Stroke,787602.0,0.041973,0.200528,0.0,0.0,0.0,0.0,1.0
HeartDiseaseorAttack,787602.0,0.091225,0.287929,0.0,0.0,0.0,0.0,1.0
PhysActivity,787602.0,0.754458,0.430408,0.0,1.0,1.0,1.0,1.0
HvyAlcoholConsump,787602.0,0.061669,0.240554,0.0,0.0,0.0,0.0,1.0
